# Build the Supplementary Material (SM)
Builds `Estimating bias in sentencing boundaries - Supplementary Material.docx` from the tables in `results_CSVs/SM/` and the figures in `plots/SM/`.
Run the "SM tables" section of `stats.ipynb` first. Table and figure numbers are assigned in order and written to `results_CSVs/SM/sm_numbering.csv`.

## Setup and helpers

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from docx import Document
from docx.enum.section import WD_ORIENT, WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt, RGBColor

ROOT     = r"C:\Users\user\OneDrive\טל\projects\justice-tal"
SM_DIR   = os.path.join(ROOT, "bias_results", "results_CSVs", "SM")
PLOT_DIR = os.path.join(ROOT, "bias_results", "plots", "SM")
POWER    = os.path.join(ROOT, "bias_results", "results_CSVs", "power_analysis_all.csv")
OUT      = os.path.join(ROOT, "Estimating bias in sentencing boundaries - Supplementary Material.docx")
PAPER_TITLE = "Uncovering Sentencing Bias with LLM-Extracted Case Facts: Evidence from Israeli Courts"

def sm(name):
    return pd.read_csv(os.path.join(SM_DIR, name), encoding="utf-8-sig")

# ---------------------------------------------------------------- labels
OFFENSE = {"cannabis": "Cannabis", "cocaine": "Cocaine", "hashish": "Hashish",
           "possession_only": "Possession", "carrying_only": "Carrying", "importing_only": "Importing"}
OUTCOME = {"min_punishment": "Lower bound", "max_punishment": "Upper bound", "is_actual_sentence": "Incarceration"}
KIND = {"impute": "Primary", "impute_0": "Lower bound = 0", "impute_6": "Lower bound = 6",
        "no_impute": "Listwise deletion", "no_personal": "No personal covariates", "unfiltered": "Unfiltered sample"}
METHOD = {"match_logit": "Logit NNM (primary)", "match_gower": "Gower distance",
          "fact_match_exact_cem": "Exact on offense vars", "manual_pairs": "Deterministic pairing"}
HEBREW = {"מחוזי": "District court", "שלום": "Magistrate court", "רווק": "single", "נשוי": "married",
          "גרוש": "divorced", "אלמן": "widowed", "ללא עבר פלילי": "none", "עבר פלילי קל": "light",
          "עבר פלילי כבד": "serious", "לא נגרם": "none", "קל": "light", "גבוה": "severe", "לא צוין": "not stated"}

TERM_LABELS = {"districtHaifa": "District: Haifa", "districtJerusalem": "District: Jerusalem",
               "districtNorth": "District: North", "districtSouth": "District: South",
               "districtTel Aviv": "District: Tel Aviv", "age": "Age", "year": "Year",
               "production offence1": "Production offense", "use offence1": "Possession / use offense",
               "importation offence1": "Trafficking / importation offense", "laboratory1": "Laboratory / growing",
               "self use1": "Personal use", "sold to agent1": "Sold to undercover agent", "total amount": "Drug quantity",
               "regret1": "Expressed regret", "confession1": "Guilty plea", "marital status: married": "Marital status: married",
               "marital status: single": "Marital status: single", "judge ethnicJewish": "Judge Jewish",
               "war period": "War period", "Treatment (Arab = 1):war period": "Treatment (Arab = 1) × war period"}

def term_label(term):
    for heb, eng in HEBREW.items():
        term = term.replace(heb, f": {eng}")
    term = (term.replace("(Intercept)", "Intercept").replace("treated", "Treatment (Arab = 1)")
                .replace("_", " ").replace("`", ""))
    return TERM_LABELS.get(term, term)

# ---------------------------------------------------------------- number formatting
def f_num(x, d=2):
    if pd.isna(x):
        return ""
    if np.isinf(x):
        return "—"          # e.g. critical Γ not defined when both groups are identical
    return f"{x:.{d}f}"

def f_p(p):
    if pd.isna(p):
        return ""
    return "< .001" if p < .001 else f"{p:.3f}".lstrip("0")

def f_p_cell(p):
    # p-value inside a "(p …; n pairs)" cell: never leave it empty
    return f_p(p) or "n/a"

def f_bse(b, se, p):
    se_txt = "—" if pd.isna(se) else f"{se:.2f}"
    return f"{b:.2f}{stars(p)} ({se_txt})"

def f_ci(lo, hi):
    return "—" if pd.isna(lo) or pd.isna(hi) else f"[{lo:.2f}, {hi:.2f}]"

from math import sqrt
from scipy.stats import fisher_exact

def binary_zero_var_p(mean_t, mean_c, n_t, n_c):
    """Fisher's exact test when a group has no variance (logistic separation gives meaningless p-values)."""
    if min(abs(mean_t - v) for v in (0, 1)) > 1e-9 and min(abs(mean_c - v) for v in (0, 1)) > 1e-9:
        return None
    n_t, n_c = int(n_t), int(n_c)
    k_t, k_c = int(round(mean_t * n_t)), int(round(mean_c * n_c))
    return fisher_exact([[k_t, n_t - k_t], [k_c, n_c - k_c]])[1]

def evalue_rr(rr):
    rr = rr if rr >= 1 else 1 / rr
    return rr + sqrt(rr * (rr - 1))

def stars(p):
    if pd.isna(p):
        return ""
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

def effect(row):
    # Difference with stars: percentage points for incarceration, months for the bounds
    if pd.isna(row.get("mean_treatment")) and pd.isna(row.get("diff")):
        return ""
    if row["outcome"] == "is_actual_sentence":
        d = 100 * (row["mean_treatment"] - row["mean_control"])
        return f"{d:+.1f} pp"
    return f"{row['diff']:+.2f}"

def means(row):
    if row["outcome"] == "is_actual_sentence":
        return f"{100 * row['mean_treatment']:.1f}", f"{100 * row['mean_control']:.1f}"
    return f_num(row["mean_treatment"]), f_num(row["mean_control"])

def std_effect(row):
    # OR for incarceration, Cohen's d for the bounds
    if row["outcome"] == "is_actual_sentence":
        orv = row.get("odds_ratio")
        if pd.isna(orv):
            orv = row.get("OR")
        return f"OR = {orv:.2f}" if pd.notna(orv) else ""
    if pd.isna(row.get("d_cohens")):
        return ""
    d = 0.0 if abs(row["d_cohens"]) < 0.005 else row["d_cohens"]
    return f"d = {d:.2f}"

# ---------------------------------------------------------------- document
doc = Document()
registry = []   # (key, label, title)
counters = {"Table": 0, "Figure": 0}

normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
normal.font.size = Pt(10)
normal.paragraph_format.space_after = Pt(4)
for lvl, size in [(1, 14), (2, 12), (3, 11)]:
    st = doc.styles[f"Heading {lvl}"]
    st.font.name = "Times New Roman"
    st.element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
    st.font.size = Pt(size)
    st.font.bold = True
    st.font.color.rgb = RGBColor(0, 0, 0)

sec = doc.sections[0]
sec.page_width, sec.page_height = Cm(21.0), Cm(29.7)
for side in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, side, Cm(2.2))

def usable_width():
    s = doc.sections[-1]
    return s.page_width - s.left_margin - s.right_margin

def set_orientation(landscape):
    current = doc.sections[-1]
    is_landscape = current.orientation == WD_ORIENT.LANDSCAPE
    if is_landscape == landscape:
        return
    s = doc.add_section(WD_SECTION.NEW_PAGE)
    s.orientation = WD_ORIENT.LANDSCAPE if landscape else WD_ORIENT.PORTRAIT
    s.page_width, s.page_height = (Cm(29.7), Cm(21.0)) if landscape else (Cm(21.0), Cm(29.7))

def heading(text, level=1):
    doc.add_heading(text, level=level)

def para(text, italic=False, size=None, bold_prefix=None, space_after=None):
    p = doc.add_paragraph()
    if bold_prefix:
        r = p.add_run(bold_prefix)
        r.bold = True
        if size:
            r.font.size = Pt(size)
    r = p.add_run(text)
    r.italic = italic
    if size:
        r.font.size = Pt(size)
    if space_after is not None:
        p.paragraph_format.space_after = Pt(space_after)

def register(kind, key, title):
    counters[kind] += 1
    label = f"{kind} SM{counters[kind]}"
    registry.append({"key": key, "label": label, "title": title})
    return label

def _borders(table):
    tbl_pr = table._tbl.tblPr
    borders = OxmlElement("w:tblBorders")
    for edge, val in [("top", "single"), ("bottom", "single"), ("insideH", "single"),
                      ("left", "nil"), ("right", "nil"), ("insideV", "nil")]:
        e = OxmlElement(f"w:{edge}")
        e.set(qn("w:val"), val)
        e.set(qn("w:sz"), "4")
        e.set(qn("w:color"), "999999" if edge == "insideH" else "000000")
        borders.append(e)
    tbl_pr.append(borders)

def _shade(cell, fill="E7E6E6"):
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), fill)
    cell._tc.get_or_add_tcPr().append(shd)

def _repeat_header(row):
    tr_pr = row._tr.get_or_add_trPr()
    h = OxmlElement("w:tblHeader")
    h.set(qn("w:val"), "true")
    tr_pr.append(h)

def add_table(key, title, df, note=None, font=8, landscape=False, widths=None):
    set_orientation(landscape)
    label = register("Table", key, title)
    cap = doc.add_paragraph()
    r = cap.add_run(label + " ")
    r.bold = True
    cap.add_run(title)
    cap.paragraph_format.keep_with_next = True
    df = df.fillna("").astype(str)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    _borders(table)
    for cell, name in zip(table.rows[0].cells, df.columns):
        cell.text = ""
        run = cell.paragraphs[0].add_run(str(name))
        run.bold = True
        run.font.size = Pt(font)
        _shade(cell)
    _repeat_header(table.rows[0])
    for values in df.itertuples(index=False):
        cells = table.add_row().cells
        for cell, v in zip(cells, values):
            cell.text = ""
            run = cell.paragraphs[0].add_run(v)
            run.font.size = Pt(font)
    for row in table.rows:
        for cell in row.cells:
            pf = cell.paragraphs[0].paragraph_format
            pf.space_after = Pt(0)
            pf.space_before = Pt(0)
    if widths:
        total = sum(widths)
        for row in table.rows:
            for cell, w in zip(row.cells, widths):
                cell.width = int(usable_width() * w / total)
    if note:
        n = doc.add_paragraph()
        r = n.add_run("Note. ")
        r.italic = True
        r.font.size = Pt(8)
        n.add_run(note).font.size = Pt(8)
    doc.add_paragraph()

def add_figure(key, title, path, note=None, max_height_cm=21.5):
    set_orientation(False)
    label = register("Figure", key, title)
    w_px, h_px = Image.open(path).size
    width = usable_width()
    height = width * h_px / w_px
    if height > Cm(max_height_cm):
        width = int(width * Cm(max_height_cm) / height)
    doc.add_picture(path, width=width)
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    cap = doc.add_paragraph()
    r = cap.add_run(label + " ")
    r.bold = True
    cap.add_run(title + (" " + note if note else ""))
    cap.runs[-1].font.size = Pt(9)

# ---------------------------------------------------------------- power diagnostics
power = pd.read_csv(POWER, encoding="utf-8-sig")
power = power[power["Method"] == "match_logit"].copy()
power["Group"] = power["Group"].astype(str).replace({"Tel": "Tel Aviv"})

def power_for(domain, analysis, offense, group, outcome):
    key = {"Ethnicity": f"{domain} - Ethnic (Jewish vs Arab)", "District": f"{domain} - District",
           "Period": f"{domain} - Period (post vs pre)", "Sector": f"{domain} - Period x Ethnic"}[analysis]
    hit = power[(power["Analysis"] == key) & (power["Type"] == offense) &
                (power["Group"] == group) & (power["Outcome"] == outcome)]
    if hit.empty:
        return "", ""
    h = hit.iloc[0]
    return (str(int(h["ReqN_perGroup"])) if pd.notna(h["ReqN_perGroup"]) else "",
            f_num(h["AchievedPower"]) if pd.notna(h["AchievedPower"]) else "")

spec = sm("sensitivity_by_specification_long.csv")
primary = spec[spec["Kind"] == "impute"].copy()
print("Setup done")

## Title and S1 Overview

In [ ]:
t = doc.add_paragraph()
t.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = t.add_run("Supplementary Material")
r.bold = True
r.font.size = Pt(16)
t2 = doc.add_paragraph()
t2.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = t2.add_run(PAPER_TITLE)
r.italic = True
r.font.size = Pt(12)
contents_anchor = doc.add_paragraph()   # the list of tables and figures is inserted here at the end

heading("S1 Overview and reproducibility", 1)
para("This Supplementary Material (SM) reports the full results behind the tables, figures and robustness checks "
     "summarized in the main text. Unless stated otherwise, results refer to the primary specification: "
     "random-forest imputation of missing personal covariates, ambiguous lower sentencing bounds coded as 3 months, "
     "and AI-powered propensity score matching (logit nearest-neighbor matching without replacement, caliper of "
     "0.2 SD of the logit of the propensity score), followed by covariate-adjusted (doubly robust) outcome models "
     "estimated on the matched samples.")
para("Conventions. Differences are treatment minus control: Arab minus Jewish defendants (ethnicity), focal district "
     "minus all other districts pooled (one-vs-rest), and post-period minus pre-period (COVID-19 onset in March 2020; "
     "war onset on 7 October 2023). Sentencing bounds are in months; incarceration differences are in percentage "
     "points (pp). p-values for district comparisons are Bonferroni-corrected (k = 30); all other p-values are "
     "nominal. * p < .05, ** p < .01, *** p < .001.")
n_primary = len(sm("balance_summary.csv"))
para("Reproducibility. The data are built in build_data.ipynb, matching and outcome models are estimated in "
     "src.ipynb (separately for the drug and weapons corpora), and all SM tables and figures are produced by the "
     "'SM tables' section of bias_results/stats.ipynb; this document is generated by bias_results/build_sm.ipynb. "
     f"Re-estimating all {n_primary} primary analyses from scratch with the final code reproduced the reported matched "
     "estimates exactly (identical group means, p-values and numbers of matched pairs). The LLM extraction prompts "
     "and code are available at https://github.com/TalKleinBgu/llm-sentencing-bias-israel.")

## S2 Data

In [ ]:
heading("S2 Analytical samples", 1)
para("Tables SM1 and SM2 describe the final analytical samples by offense type and defendant ethnicity. Personal "
     "characteristics are shown as observed, before imputation, together with the share of unknown values.")

for domain in ["Drugs", "Weapons"]:
    desc = pd.read_csv(os.path.join(SM_DIR, f"descriptive_statistics_{domain.lower()}.csv"),
                       header=[0, 1], index_col=0, encoding="utf-8-sig")
    desc.columns = [f"{OFFENSE.get(o, o)}: {g}" for o, g in desc.columns]
    desc = desc.reset_index().rename(columns={"index": "Characteristic", desc.index.name or "index": "Characteristic"})
    desc = desc.rename(columns={desc.columns[0]: "Characteristic"})
    add_table(f"desc_{domain.lower()}", f"Descriptive statistics of the {domain.lower()} corpus by offense type and defendant ethnicity",
              desc, font=7, landscape=True, widths=[3.2] + [1] * (len(desc.columns) - 1),
              note="All = all defendants of the offense type, including those whose ethnicity could not be classified. "
                   "Personal characteristics are observed values before imputation. "
                   + ("'Bullets present, count not stated' are cases in which the extracted count was -1; these are set to zero in the propensity score models."
                      if domain == "Weapons" else "Drug quantity in grams. Only the three drug types analyzed separately are shown."))

## S3 LLM extraction validation

In [ ]:
set_orientation(False)
heading("S3 LLM extraction validation", 1)
para("Separate validation samples of 125 verdicts per corpus were annotated independently by two human coders; "
     "disagreements were resolved by discussion. Table SM3 reports inter-coder agreement. Tables SM4 and SM5 "
     "compare the LLM extraction with the consensus annotation, overall and separately for Jewish and Arab "
     "(non-Jewish) defendants, with 95% bootstrap confidence intervals.")

VAR_LABELS = {
    "Guilty Plea": "Guilty plea", "confession": "Guilty plea", "responsabillity": "Responsibility",
    "regret": "Expressed regret", "marital_status": "Marital status", "self_use": "Personal use",
    "laboratory": "Laboratory / growing", "role": "Role in offense", "sold_to_agent": "Sold to undercover agent",
    "production_offence": "Offense: production", "possession_offence": "Offense: possession",
    "trafficking_offence": "Offense: trafficking", "Criminal_record": "Criminal record",
    "has_independent_offenses": "Independent offenses", "section_144a": "Offense: possession (144a)",
    "section_144b": "Offense: carrying (144b)", "section_144b2": "Offense: importing (144b2)",
    "section_144b3": "Offense: 144b3", "escape_attempt": "Escape attempt", "premeditation": "Premeditation",
    "undercover_sale": "Undercover sale", "is_actual_sentence": "Incarceration (sentence type)",
    "actual_damage": "Actual damage", "weapon_acquisition": "Weapon acquisition", "weapon_motive": "Weapon motive",
    "weapon_status": "Weapon status", "weapon_use": "Weapon use", "weapon_storage": "Weapon storage",
    "age": "Age", "punishment_months": "Sentence length (months)", "Lower Bounds": "Range lower bound",
    "Upper Bounds": "Range upper bound", "min_punishment": "Range lower bound", "max_punishment": "Range upper bound",
    "drug_quantity_log": "Drug quantity (log)", "magazines": "Magazines", "bullets": "Bullets",
    "weapon_counts_combined": "Weapon counts",
}
validation = sm("llm_validation_full.csv")
validation["Variable"] = validation["Variable"].map(lambda v: VAR_LABELS.get(v, v))

# Per-variable inter-coder values are not in the repository; report the means (as in the main text).
MEAN_AGREEMENT = {("Drugs", "Categorical"): 0.78, ("Drugs", "Continuous"): 0.79,
                  ("Weapons", "Categorical"): 0.76, ("Weapons", "Continuous"): 0.82}
kappa_rows = []
for domain in ["Drugs", "Weapons"]:
    v = validation[validation["Domain"] == domain]
    for typ, measure, label in [("Categorical", "Mean Cohen's κ", "All categorical and binary variables"),
                                ("Continuous", "Mean ICC", "All continuous variables")]:
        kappa_rows.append({"Corpus": domain, "Variable": f"{label} ({int((v['Type'] == typ).sum())})",
                           "Measure": measure, "Agreement": f"{MEAN_AGREEMENT[(domain, typ)]:.2f}"})
add_table("intercoder", "Inter-coder agreement by variable (two independent coders, 125 verdicts per corpus)",
          pd.DataFrame(kappa_rows), font=8, widths=[1, 2.5, 1, 1],
          note="Mean agreement between the two coders across the variables of each type (125 verdicts per corpus); "
               "κ = Cohen's kappa for categorical and binary variables, ICC = intraclass correlation coefficient for "
               "continuous variables. Per-variable values are available from the authors on request.")

for domain in ["Drugs", "Weapons"]:
    v = validation[validation["Domain"] == domain].copy()
    table = v[["Variable", "Type", "Metric", "n", "All", "Jewish", "Arab / non-Jewish", "CIs overlap"]]
    table = table.rename(columns={"Arab / non-Jewish": "Arab" if domain == "Weapons" else "Non-Jewish",
                                  "CIs overlap": "CIs overlap (Jewish vs. " + ("Arab)" if domain == "Weapons" else "non-Jewish)")})
    add_table(f"valid_{domain.lower()}", f"LLM extraction accuracy against human coding, {domain.lower()} corpus, overall and by defendant ethnicity",
              table, font=7.5, widths=[2.4, 1.2, 0.7, 0.6, 2, 2, 2, 1.1],
              note="F1 (macro) for categorical variables and mean absolute error (MAE) for continuous variables, with 95% bootstrap "
                   "confidence intervals in brackets. n = number of annotated cases ("
                   + ("for weapon counts: case × weapon-type cells). " if domain == "Weapons" else "for drug quantity: case × drug-type cells). ")
                   + ("Weapon status is a descriptive variable that is not included in the propensity score model."
                      if domain == "Weapons" else "In the drugs corpus the minority group is defined as non-Jewish defendants."))

## S4 Matching diagnostics

In [ ]:
heading("S4 Matching diagnostics", 1)
heading("S4.1 Deterministic pairing", 2)
para("In the deterministic pairing specification (Section 3.3.1d of the main text), cases are sorted in ascending order "
     "by the variables in Table SM6, in the order shown; each Arab defendant is then paired with the nearest available "
     "Jewish defendant in the sorted order, and both cases are removed from the pool.")
key = sm("deterministic_pairing_sort_key.csv")
SORT_LABELS = {"total_amount": "Drug quantity", "laboratory": "Laboratory / growing", "role": "Role in offense",
               "sold_to_agent": "Sold to undercover agent", "importation_offence": "Offense: trafficking / importation",
               "production_offence": "Offense: production", "use_offence": "Offense: possession / use",
               "self_use": "Personal use", "year": "Year", "Court": "Court level", "district": "District",
               "Location": "Court location", "Judges_Ethnic": "Judge ethnicity", "judge_ethnic": "Judge ethnicity",
               "marital_status": "Marital status", "age": "Age", "Date": "Verdict date", "magazines": "Magazines",
               "bullets": "Bullets", "handgun": "Handguns", "submachine_gun": "Submachine guns",
               "assault_rifle": "Assault rifles", "improvised_firearm": "Improvised firearms", "grenade": "Grenades",
               "explosive_charge": "Explosive charges", "actual_damage": "Actual damage", "premeditation": "Premeditation",
               "escape_attempt": "Escape attempt", "undercover_sale": "Undercover sale",
               "has_independent_offenses": "Independent offenses"}
key["Variable"] = key["Variable"].map(lambda v: SORT_LABELS.get(v, v))
wide = key.pivot(index="Order", columns="Domain", values="Variable").reset_index().fillna("")
add_table("sortkey", "Hierarchical sort key used for deterministic pairing", wide, font=8.5, widths=[0.6, 2, 2])

heading("S4.2 Covariate balance and propensity score overlap", 2)
para("Table SM7 summarizes covariate balance for every primary analysis: the largest absolute standardized mean "
     "difference (ASMD) across covariates before and after matching, and the covariates whose ASMD remained above 0.25 "
     "after matching. The full balance for each analysis is shown in the love plots that follow (one figure per "
     "offense type and family of analyses), and boxplots of the propensity score distributions of the treatment and "
     "control groups before and after matching show the common support of each analysis.")
bal = sm("balance_summary.csv")
bal["Offense"] = bal["Offense"].map(OFFENSE)
bal["Analysis"] = bal["Analysis"].map({"ethnicity": "Ethnicity", "district": "District (one vs rest)",
                                       "period": "Period", "sector": "Period within ethnic group"})
bal = bal.rename(columns={"max |ASMD| before": "Max ASMD before", "max |ASMD| after": "Max ASMD after",
                          "covariates > 0.25 after": "n > 0.25", "which": "Covariates above 0.25 after matching"})
add_table("balance", "Covariate balance before and after matching, all primary analyses", bal, font=7, landscape=True,
          widths=[0.8, 0.9, 1.3, 1.6, 0.8, 0.8, 0.6, 5],
          note="ASMD = absolute standardized mean difference; values below 0.25 indicate adequate balance. Binary covariates are "
               "counted once. The propensity score itself is excluded from the maximum.")

FAMILY = {"ethnicity": "ethnicity (Arab vs. Jewish)", "district": "one-vs-rest district comparisons",
          "period": "COVID-19 and war periods", "sector": "war period within ethnic group"}
for domain, offenses in [("drugs", ["cannabis", "cocaine", "hashish"]),
                         ("weapons", ["possession_only", "carrying_only", "importing_only"])]:
    for offense in offenses:
        for family in ["ethnicity", "district", "period", "sector"]:
            path = os.path.join(PLOT_DIR, f"love_{domain}_{offense}_{family}.png")
            if os.path.exists(path):
                add_figure(f"love_{domain}_{offense}_{family}",
                           f"Covariate balance before and after AI-powered matching: {OFFENSE[offense].lower()} ({domain}), {FAMILY[family]}.",
                           path, note="Open circles: before matching; filled circles: after matching; dashed line: ASMD = 0.25.")
for domain, offenses in [("drugs", ["cannabis", "cocaine", "hashish"]),
                         ("weapons", ["possession_only", "carrying_only", "importing_only"])]:
    for offense in offenses:
        path = os.path.join(PLOT_DIR, f"overlap_{domain}_{offense}.png")
        if os.path.exists(path):
            add_figure(f"overlap_{domain}_{offense}",
                       f"Propensity score distributions before and after matching: {OFFENSE[offense].lower()} ({domain}).",
                       path, note="Each panel shows one primary analysis: boxplots of the estimated propensity score for the control "
                                  "(gold) and treatment (blue) groups among all cases before matching and among matched cases after "
                                  "matching; the panel title reports how many treated cases were matched.")

## S5 Ethnic disparities

In [ ]:
heading("S5 Ethnic disparities", 1)
para("Table SM8 reports the full results behind Figures 1 and 3 of the main text: group outcomes, differences and "
     "tests under the naïve, standard and AI-powered specifications for all six offense types. Table SM9 extends "
     "Table 6 to all outcomes and both domains; Table SM10 reports the full covariate-adjusted models summarized in "
     "Table 7; Table SM11 reports the ethnicity × war interaction models.")

three = sm("ethnicity_three_levels.csv")
rows = []
for _, r in three.iterrows():
    a, j = means(r)
    ci = "" if r["outcome"] == "is_actual_sentence" or pd.isna(r["ci_low"]) else f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]"
    req, pw = power_for(r["Domain"], "Ethnicity", r["Offense"], "-", r["outcome"]) if r["Level"] == "AI-powered" else ("", "")
    p_val, es = r["p_value"], std_effect(r)
    if r["outcome"] == "is_actual_sentence" and pd.notna(p_val):
        pf = binary_zero_var_p(r["mean_treatment"], r["mean_control"], r["n_arab"], r["n_jewish"])
        if pf is not None and r["mean_treatment"] != r["mean_control"]:
            p_val, es = pf, "—"
    rows.append({"Domain": r["Domain"], "Offense": OFFENSE[r["Offense"]], "Outcome": OUTCOME[r["outcome"]],
                 "Specification": r["Level"], "Arab": a, "Jewish": j, "Difference": effect(r) + stars(p_val),
                 "95% CI": ci, "Effect size": es, "p": f_p(p_val),
                 "n (Arab / Jewish)": f"{int(r['n_arab'])} / {int(r['n_jewish'])}", "Req. n": req, "Power": pw})
add_table("eth3", "Ethnic disparities under three levels of covariate adjustment, all offense types",
          pd.DataFrame(rows), font=7, landscape=True, widths=[0.8, 0.9, 1, 1, 0.7, 0.7, 1, 1.2, 0.9, 0.6, 1, 0.6, 0.6],
          note="Arab and Jewish columns are group means (months) or incarceration rates (%). Naïve = before matching; "
               "Standard = matching on the covariates available in administrative data; AI-powered = matching on all LLM-extracted "
               "covariates (primary). n = defendants (before matching) or matched pairs (after matching). Req. n = matched pairs per "
               "group required for 80% power (one-sided α = .05) to detect the pre-specified smallest effect of interest "
               "(6 months for the bounds; h = 0.40 for incarceration); Power = achieved power at the observed n (primary specification only). "
               "When one group has no variance in incarceration (e.g., all Jewish carrying defendants were incarcerated), the p-value is "
               "from Fisher's exact test. Four weapons cases (three carrying, one importing) in which the presiding judge's ethnicity "
               "could not be classified are excluded from the matching analyses, so the naive n differs slightly from Table SM2.")

rob = sm("robustness_matching_long.csv")
rob = rob[rob["Analysis"] == "Ethnicity"]
rows = []
for (domain, offense, outcome), g in rob.groupby(["Domain", "Offense", "outcome"], sort=False):
    row = {"Domain": domain, "Offense": OFFENSE[offense], "Outcome": OUTCOME[outcome]}
    for method, label in METHOD.items():
        m = g[g["Method"] == method]
        if m.empty:
            row[label] = ""
            continue
        r = m.iloc[0]
        es = std_effect(r)
        n = r["n_treatment_unique"] if pd.notna(r.get("n_treatment_unique")) else r.get("n_pairs")
        row[label] = f"{effect(r)}{stars(r['p_value'])}" + (f", {es}" if es else "") + f" (p {f_p_cell(r['p_value'])}" + (f"; {int(n)} pairs)" if pd.notna(n) else ")")
    rows.append(row)
add_table("eth_robust", "Robustness of the ethnic disparities to the matching specification (extends Table 6)",
          pd.DataFrame(rows), font=7, landscape=True, widths=[0.8, 0.9, 1, 2.6, 2.6, 2.6, 2.6],
          note="Each cell: difference (months, or percentage points for incarceration), effect size (OR for incarceration), "
               "p-value and number of matched pairs. See Section 3.3.1 of the main text for the four specifications.")

fm = sm("full_model_cannabis.csv")
fm["cell"] = fm.apply(lambda r: f"{r['B']:.2f}{stars(r['p_value'])} ({r['SE']:.2f})", axis=1)
fm["col"] = fm["Model"].str.replace(" matching", "") + ": " + fm["outcome"].map(OUTCOME)
order = [f"{m}: {o}" for m in ["Before", "Standard", "AI-powered"] for o in OUTCOME.values()]
table = fm.pivot_table(index="term", columns="col", values="cell", aggfunc="first").reindex(columns=order)
terms = list(dict.fromkeys(["(Intercept)", "treated"] + fm["term"].tolist()))
table = table.reindex(terms).reset_index()
table["term"] = table["term"].map(term_label)
r2 = fm.groupby("col")["R_squared"].first().reindex(order).map(lambda x: f_num(x))
table = pd.concat([table, pd.DataFrame([["R² / pseudo-R²"] + r2.tolist()], columns=table.columns)], ignore_index=True)
table = table.rename(columns={"term": "Term"})
add_table("fullmodel", "Covariate-adjusted models of sentencing outcomes, cannabis ethnicity analysis (full version of Table 7)",
          table, font=6.5, landscape=True, widths=[2.4] + [1] * 9,
          note="Unstandardized coefficients with standard errors in parentheses: OLS for the bounds (months) and logistic regression "
               "for incarceration (log-odds). Models after matching include the war-period indicator and its interaction with ethnicity, "
               "so the treatment coefficient is the Arab–Jewish difference before the war. Empty cells: term not in that model. "
               "Reference categories: Center District, District court, no criminal record, divorced, Arab judge. Coefficients above |10| "
               "in the incarceration models (e.g., District: North, Guilty plea, Judge Jewish) reflect quasi-complete separation and "
               "should not be interpreted.")

inter = sm("ethnicity_war_interaction.csv")
inter["Offense"] = inter["Offense"].map(OFFENSE)
inter["Outcome"] = inter["outcome"].map(OUTCOME)
inter["Term"] = inter["term"].map({"treated": "Arab (pre-war gap)", "war_period": "War period",
                                   "treated:war_period": "Arab × war period"})
inter["B (SE)"] = inter.apply(lambda r: f"{r['B']:.2f}{stars(r['p_value'])} ({r['SE']:.2f})", axis=1)
inter["95% CI"] = inter.apply(lambda r: f"[{r['CI_low']:.2f}, {r['CI_high']:.2f}]", axis=1)
inter["p"] = inter["p_value"].map(f_p)
add_table("interaction", "Ethnicity × war-period interaction models on the matched ethnicity samples",
          inter[["Domain", "Offense", "Outcome", "Term", "B (SE)", "95% CI", "p"]], font=7.5,
          widths=[0.8, 0.9, 1, 1.4, 1.2, 1.4, 0.6],
          note="Doubly robust models estimated on the AI-powered matched samples. Bounds: OLS coefficients in months; incarceration: "
               "logistic coefficients (log-odds). Estimates with very large coefficients and standard errors (hashish and weapons "
               "incarceration) reflect complete or quasi-complete separation and should not be interpreted. The hashish lower-bound "
               "interaction rests on few post-war matched cases.")

## S6 Geographic disparities

In [ ]:
heading("S6 Geographic disparities (one-vs-rest)", 1)
para("Tables SM12 and SM13 report every one-vs-rest comparison: the focal district is compared with all other districts "
     "pooled after AI-powered matching. p-values are Bonferroni-corrected across the 30 district × offense-type comparisons; "
     "power diagnostics and sensitivity statistics are reported for every comparison.")

def district_rows(domain):
    d = primary[(primary["Domain"] == domain) & (primary["Analysis"] == "District (one vs rest)")].copy()
    rows = []
    for _, r in d.iterrows():
        district = r["Comparison"].replace(" vs rest", "")
        a, j = means(r)
        req, pw = power_for(domain, "District", r["Offense"], district, r["outcome"])
        rows.append({"Offense": OFFENSE[r["Offense"]], "District": district, "Outcome": OUTCOME[r["outcome"]],
                     "District mean": a, "Rest mean": j, "Difference": effect(r) + stars(r["p_bonferroni"]),
                     "Effect size": std_effect(r), "p": f_p(r["p_value"]), "p (Bonf.)": f_p(r["p_bonferroni"]),
                     "Pairs": str(int(r["n_treatment_unique"])) if pd.notna(r["n_treatment_unique"]) else "",
                     "Req. n": req, "Power": pw,
                     "Γ": f_num(r["rosenbaum_gamma"], 1), "E-value": f_num(r["e_value_point"])})
    return pd.DataFrame(rows)

NOTE_DISTRICT = ("Difference = focal district minus all other districts pooled, after matching (months, or percentage points for "
                 "incarceration). Stars refer to the Bonferroni-corrected p-value (k = 30). Req. n = matched pairs per group required "
                 "for 80% power to detect the pre-specified smallest effect of interest (6 months; h = 0.40); Power = achieved power. "
                 "Γ = critical Rosenbaum Γ; E-value for the point estimate. Blank cells or —: not estimable (e.g., no variation in incarceration).")
for domain in ["Drugs", "Weapons"]:
    add_table(f"district_{domain.lower()}", f"One-vs-rest district comparisons, {domain.lower()} corpus",
              district_rows(domain), font=7, landscape=True,
              widths=[0.9, 0.9, 1, 0.8, 0.8, 1, 0.9, 0.6, 0.7, 0.5, 0.6, 0.6, 0.5, 0.7], note=NOTE_DISTRICT)

## S7 Temporal stability

In [ ]:
heading("S7 Temporal stability", 1)
para("Tables SM14 and SM15 report the COVID-19 and war-period comparisons for all offense types; Table SM16 reports the "
     "within-group comparisons of the war period estimated separately for Arab and Jewish defendants.")

def period_rows(domain, analysis):
    d = primary[(primary["Domain"] == domain) & (primary["Analysis"] == analysis)].copy()
    rows = []
    for _, r in d.iterrows():
        a, j = means(r)
        if analysis == "Period":
            group = "corona" if r["Comparison"].startswith("COVID") else "war"
            req, pw = power_for(domain, "Period", r["Offense"], group, r["outcome"])
            label = {"Period": "COVID-19" if group == "corona" else "War"}
        else:
            sector = r["Comparison"].split(", ")[-1]
            req, pw = power_for(domain, "Sector", r["Offense"], sector, r["outcome"])
            label = {"Group": sector}
        rows.append({"Offense": OFFENSE[r["Offense"]], **label, "Outcome": OUTCOME[r["outcome"]],
                     "Post": a, "Pre": j, "Difference": effect(r) + stars(r["p_value"]),
                     "Effect size": std_effect(r), "p": f_p(r["p_value"]),
                     "Pairs": str(int(r["n_treatment_unique"])) if pd.notna(r["n_treatment_unique"]) else "",
                     "Req. n": req, "Power": pw})
    return pd.DataFrame(rows)

NOTE_PERIOD = ("Difference = post-period minus pre-period after matching (months, or percentage points for incarceration). "
               "COVID-19 onset: March 2020; war onset: 7 October 2023. Adjudication year is excluded from the matching covariates. "
               "Req. n / Power as in Table SM8.")
for domain in ["Drugs", "Weapons"]:
    add_table(f"period_{domain.lower()}", f"Sentencing outcomes before and after the COVID-19 pandemic and the October 2023 war, {domain.lower()} corpus",
              period_rows(domain, "Period"), font=7.5, widths=[0.9, 0.8, 1, 0.8, 0.8, 1, 0.9, 0.6, 0.6, 0.6, 0.6], note=NOTE_PERIOD)

sector = pd.concat([period_rows(d, "Period within sector").assign(Domain=d) for d in ["Drugs", "Weapons"]], ignore_index=True)
sector = sector[["Domain"] + [c for c in sector.columns if c != "Domain"]]
add_table("sector", "War-period comparisons within ethnic group (Figure 9 of the main text)", sector, font=7.5,
          widths=[0.8, 0.9, 0.8, 1, 0.8, 0.8, 1, 0.9, 0.6, 0.6, 0.6, 0.6],
          note="Matching is performed separately within Arab and Jewish defendants; ethnicity is therefore not a covariate. " + NOTE_PERIOD)

## S8 Covariate-adjusted treatment effects

In [ ]:
heading("S8 Covariate-adjusted (doubly robust) treatment effects", 1)
para("Table SM17 reports the treatment coefficient of the covariate-adjusted outcome model estimated on each matched sample, "
     "for every primary analysis. These estimates complement the matched comparisons in Tables SM8 and SM12–SM16.")
adj = sm("adjusted_treatment_effects.csv")
adj["Offense"] = adj["Offense"].map(OFFENSE)
adj["Outcome"] = adj["outcome"].map(OUTCOME)
adj["B (SE)"] = adj.apply(lambda r: f_bse(r["B"], r["SE"], r["p_value"]), axis=1)
adj["95% CI"] = adj.apply(lambda r: f_ci(r["CI_low"], r["CI_high"]), axis=1)
adj["p"] = adj["p_value"].map(f_p)
adj["Analysis"] = adj["Analysis"].replace({"Period within sector": "Period within ethnic group"})
add_table("adjusted", "Treatment effects from covariate-adjusted models on the matched samples, all primary analyses",
          adj[["Domain", "Offense", "Analysis", "Comparison", "Outcome", "B (SE)", "95% CI", "p"]], font=7,
          widths=[0.8, 0.9, 1.5, 1.7, 1, 1.2, 1.3, 0.6],
          note="OLS coefficients (months) for the bounds and logistic coefficients (log-odds) for incarceration. For the drug ethnicity "
               "analyses the models include the war-period interaction (Table SM11), so the coefficient is the pre-war difference. "
               "p-values are nominal. Very large coefficients reflect separation in near-universal incarceration.")

## S9 Sensitivity to unmeasured confounding

In [ ]:
heading("S9 Sensitivity to unmeasured confounding", 1)
para("For every statistically significant finding in the primary specification we computed Rosenbaum bounds (Rosenbaum 2002) "
     "and E-values (VanderWeele and Ding 2017). The critical Γ is the smallest value of the sensitivity parameter at which "
     "the Rosenbaum bound on the p-value exceeds .05 (grid of 0.1); an unobserved covariate would have to change the odds of "
     "treatment by a factor of Γ within matched pairs to explain the finding away. Binary outcomes use the McNemar-type bound "
     "(binarysens) and continuous outcomes the signed-rank bound (psens). The E-value is the minimum strength of association, on "
     "the risk-ratio scale, that an unmeasured confounder would need with both treatment and outcome to explain away the point "
     "estimate (point) or to move the confidence interval to include the null (CI); continuous effects are converted via the "
     "standardized mean difference.")
sens = sm("sensitivity_rosenbaum_evalues.csv")
# The pipeline leaves eval_ci empty when OR < 1; recompute it from the upper OR bound (RR ≈ sqrt(OR), common outcome)
_dist = pd.concat([pd.read_csv(os.path.join(ROOT, "bias_results", "results_CSVs", f), encoding="utf-8-sig")
                   for f in ["drugs_district_all_results.csv", "weapon_district_all_results.csv"]], ignore_index=True)
_dist["District"] = _dist["District"].replace({"Tel": "Tel Aviv"})
def _eval_ci(r):
    if pd.notna(r["eval_ci"]) or r["outcome"] != "is_actual_sentence" or not (r["odds_ratio"] < 1):
        return r["eval_ci"]
    hit = _dist[(_dist["Type"] == r["Offense"]) & (_dist["District"] == r["Comparison"].replace(" vs rest", "")) &
                (_dist["outcome"] == r["outcome"])]
    if hit.empty or not (hit.iloc[0]["or_ci_high"] < 1):
        return r["eval_ci"]
    return evalue_rr(sqrt(hit.iloc[0]["or_ci_high"]))
sens["eval_ci"] = sens.apply(_eval_ci, axis=1)
rows = []
for _, r in sens.iterrows():
    eff = (f"OR = {r['odds_ratio']:.2f}" if r["outcome"] == "is_actual_sentence" and pd.notna(r["odds_ratio"])
           else f"Δ = {r['diff']:+.2f} months")
    p_used = r["p_bonferroni"] if str(r["Analysis"]).startswith("District") else r["p_value"]
    rows.append({"Domain": r["Domain"], "Offense": OFFENSE[r["Offense"]], "Analysis": r["Analysis"].replace("Period within sector", "Period within group"),
                 "Comparison": r["Comparison"], "Outcome": OUTCOME[r["outcome"]], "Effect": eff, "p": f_p(p_used),
                 "Pairs": str(int(r["n_treatment_unique"])), "Critical Γ": f_num(r["rosenbaum_gamma"], 1),
                 "E-value (point)": f_num(r["e_value_point"]), "E-value (CI)": f_num(r["eval_ci"])})
add_table("sensitivity", "Rosenbaum bounds and E-values for all statistically significant findings (extends Table 8)",
          pd.DataFrame(rows), font=7, landscape=True, widths=[0.8, 1, 1.4, 1.8, 1, 1.3, 0.6, 0.5, 0.7, 0.8, 0.8],
          note="p = Bonferroni-corrected for district comparisons, nominal otherwise. Findings based on fewer than 15 matched pairs "
               "(weapons importing, war period; Jewish defendants in weapons carrying, war period) are shown for completeness but are "
               "not interpreted. Blank critical Γ: not computable.")

## S10 Robustness checks

In [ ]:
heading("S10 Robustness checks", 1)
para("Tables SM19 and SM20 re-estimate every primary comparison under alternative data specifications: ambiguous lower bounds "
     "coded as 0 or 6 months instead of 3; listwise deletion of cases with missing personal covariates instead of imputation; "
     "exclusion of age, marital status and criminal record from the covariate set; and, for the ethnic incarceration gap, the "
     "unfiltered sample that also includes verdicts without a declared sentencing range. Table SM21 reports the significant "
     "district and temporal findings under Gower distance and exact matching, and Table SM22 examines selection into the "
     "filtered sample.")

def spec_cell(r):
    if pd.isna(r["diff"]) and pd.isna(r.get("mean_treatment")):
        return ""
    n = f", {int(r['n_treatment_unique'])}" if pd.notna(r["n_treatment_unique"]) else ""
    return f"{effect(r)}{stars(r['p_used'])} ({f_p_cell(r['p_used'])}{n})"

for domain in ["Drugs", "Weapons"]:
    d = spec[spec["Domain"] == domain].copy()
    d["cell"] = d.apply(spec_cell, axis=1)
    d["Offense"] = d["Offense"].map(OFFENSE)
    d["Outcome"] = d["outcome"].map(OUTCOME)
    d["Analysis"] = d["Analysis"].replace({"Period within sector": "Period within group"})
    table = (d.pivot_table(index=["Offense", "Analysis", "Comparison", "Outcome"], columns="Kind", values="cell",
                           aggfunc="first", sort=False)
             .reindex(columns=[k for k in KIND if k in d["Kind"].unique()]).rename(columns=KIND).reset_index())
    add_table(f"spec_{domain.lower()}", f"Sensitivity of all primary comparisons to the data specification, {domain.lower()} corpus",
              table, font=6, landscape=True, widths=[0.8, 1.2, 1.6, 0.9] + [1.5] * (len(table.columns) - 4),
              note="Each cell: difference (months, or percentage points for incarceration) with stars, followed by the p-value and the number "
                   "of matched pairs (n/a: p-value not estimable because all cases were incarcerated). p-values are Bonferroni-corrected for district comparisons. The unfiltered sample is analyzed for the "
                   "ethnic incarceration gap only. Blank cells: comparison not estimable in that specification.")

rob = sm("robustness_matching_long.csv")
rob = rob[rob["Analysis"] != "Ethnicity"]
rows = []
for (domain, offense, comparison, outcome), g in rob.groupby(["Domain", "Offense", "Comparison", "outcome"], sort=False):
    row = {"Domain": domain, "Offense": OFFENSE[offense], "Comparison": comparison, "Outcome": OUTCOME[outcome]}
    for method in ["match_logit", "match_gower", "fact_match_exact_cem"]:
        m = g[g["Method"] == method]
        if m.empty:
            row[METHOD[method]] = ""
            continue
        r = m.iloc[0]
        row[METHOD[method]] = f"{effect(r)}{stars(r['p_used'])} (p {f_p_cell(r['p_used'])}; {int(r['n_treatment_unique'])} pairs)"
    rows.append(row)
add_table("robust_dt", "Robustness of the significant district and temporal findings to the matching specification",
          pd.DataFrame(rows), font=7.5, landscape=True, widths=[0.8, 1, 1.6, 1, 2.2, 2.2, 2.2],
          note="p-values are Bonferroni-corrected for district comparisons and nominal for period comparisons. Gower distance: offense "
               "variables weighted 3, caliper at the 90th percentile of nearest-neighbor distances; exact matching on offense variables "
               "with Mahalanobis matching on quantities.")

rates = sm("exclusion_rates_by_ethnicity.csv")
sel = sm("selection_into_filtered_sample.csv")
rows = []
for domain in ["Drugs", "Weapons"]:
    rt = rates[rates["Domain"] == domain].set_index("ethnic")
    for _, r in sel[sel["Domain"] == domain].iterrows():
        rows.append({"Domain": domain,
                     "Unfiltered n": str(int(r["n"])), "Excluded n": str(int(r["n_excluded"])),
                     "Excluded, Arab %": f"{100 * rt.loc['Arab', 'excluded_share']:.1f}",
                     "Excluded, Jewish %": f"{100 * rt.loc['Jewish', 'excluded_share']:.1f}",
                     "Predictor": {"arab": "Arab defendant", "np.log1p(total_amount)": "Drug quantity (log)",
                                   "np.log1p(bullets)": "Bullets (log)", "np.log1p(weapon_count)": "Weapons seized (log)"}[r["term"]],
                     "OR [95% CI]": f"{r['OR']:.2f} [{r['OR_ci_low']:.2f}, {r['OR_ci_high']:.2f}]", "p": f_p(r["p_value"])})
add_table("selection", "Selection into the filtered sample: exclusion of verdicts without a declared sentencing range",
          pd.DataFrame(rows), font=8, widths=[0.8, 0.8, 0.8, 1, 1, 1.6, 1.6, 0.6],
          note="Logistic regression of exclusion (no declared range) on defendant ethnicity, offense type, severity (drug quantity; "
               "bullets and weapons seized), criminal record, court level and district, within the unfiltered sample of Arab and "
               "Jewish defendants (drugs corpus: cannabis, cocaine and hashish only). The ethnic incarceration gap in the unfiltered samples is reported in Tables SM19 and SM20.")

## S11 Power analysis

In [ ]:
set_orientation(False)
heading("S11 Power analysis", 1)
para("Power was assessed for a pre-specified smallest effect size of interest rather than for the observed effects, because "
     "power computed from an observed effect is a direct function of its p-value (Hoenig and Heisey 2001). The smallest effect "
     "of interest is a 6-month difference in a sentencing bound, standardized by the pooled standard deviation of each "
     "comparison (Cohen's d), and Cohen's h = 0.40 for incarceration (for example, an increase from 76% to about 90% at the "
     "cannabis baseline). For each comparison we report the number of matched pairs per group required to detect this effect "
     "with 80% power in a one-sided test at α = .05, and the power achieved at the observed number of pairs (columns Req. n and "
     "Power in Tables SM8 and SM12–SM16). Comparisons below the requirement are interpreted as inconclusive rather than as "
     "evidence of no effect.")

## S12 Distribution of the quantity covariates

In [ ]:
heading("S12 Distribution of the quantity covariates", 1)
para("Drug quantities and weapon, bullet and magazine counts are highly right-skewed, with a few very large seizures "
     "(for example, one weapons-carrying case involves 579 improvised handguns). To limit the influence of such values on the "
     "propensity score model, these covariates were log-transformed: log(kg + 0.001) for drug quantity and log(1 + x) for "
     "counts, with an unstated bullet count (coded -1 by the extraction when a loaded magazine is mentioned without a number) "
     "set to zero. Table SM23 and Figure SM29 show that the transformation removes most of the skewness.")
sk = sm("quantity_skewness.csv")
sk["Offense"] = sk["Offense"].map(OFFENSE)
sk["Max"] = sk["Max"].map(lambda v: f"{v:,.0f}")
sk["Unstated (-1), %"] = sk["Unstated (-1), %"].map(lambda v: "" if pd.isna(v) else f"{v:.1f}")
add_table("skew", "Skewness of the quantity covariates before and after the log transformation", sk, font=8,
          widths=[0.8, 0.9, 1.6, 0.5, 1.6, 0.8, 0.9, 0.9, 1],
          note="Median [IQR] and maximum of the untransformed variable (bullets: over stated counts). Skewness of the raw variable "
               "(unstated counts set to zero) and of the transformed variable entering the propensity score model.")
add_figure("skew_fig", "Distribution of the quantity covariates before (grey) and after (blue) the log transformation.",
           os.path.join(PLOT_DIR, "quantity_distributions.png"),
           note="Weapons importing (n = 71) is summarized in Table SM23 only.")

## List of tables and figures, numbering map, save

In [ ]:
# List of tables and figures, inserted after the title
items = [("List of supplementary tables", True)]
items += [(f"{x['label']}. {x['title']}", False) for x in registry if x["label"].startswith("Table")]
items += [("List of supplementary figures", True)]
items += [(f"{x['label']}. {x['title']}", False) for x in registry if x["label"].startswith("Figure")]
anchor = contents_anchor._p
for text, bold in reversed(items):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold
    run.font.size = Pt(11 if bold else 9)
    p.paragraph_format.space_after = Pt(1)
    anchor.addnext(p._p)

# The section texts refer to table numbers; make sure the numbering matches them
EXPECTED_TABLES = ["desc_drugs", "desc_weapons", "intercoder", "valid_drugs", "valid_weapons", "sortkey", "balance",
                   "eth3", "eth_robust", "fullmodel", "interaction", "district_drugs", "district_weapons",
                   "period_drugs", "period_weapons", "sector", "adjusted", "sensitivity", "spec_drugs", "spec_weapons",
                   "robust_dt", "selection", "skew"]
tables = [x["key"] for x in registry if x["label"].startswith("Table")]
assert tables == EXPECTED_TABLES, f"Table order changed — update the section texts: {tables}"

pd.DataFrame(registry).to_csv(os.path.join(SM_DIR, "sm_numbering.csv"), index=False, encoding="utf-8-sig")
doc.save(OUT)
print(f"Saved {OUT}")
print(f"{counters['Table']} tables, {counters['Figure']} figures")
print(pd.DataFrame(registry)[["label", "key"]].to_string(index=False))